In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 4 (orden recomendado): Creación de ventanas deslizantes de 72h.
Entrada: datos codificados (salida del Código 3) en las carpetas:
    - encoded/ml/by_station/
    - encoded/ml/global/
    - encoded/dl/by_station/
    - encoded/dl/global/
Salida: ventanas en formato numpy arrays (.npy) para modelos ML (2D) y DL (3D),
        además de archivos JSON con la lista de features.
Estructura de salida:
    windows/
        by_station/
            ml/
                estacion1_X.npy, estacion1_y.npy, features.json
                estacion2_X.npy, estacion2_y.npy, features.json
                ...
            dl/
                estacion1_X.npy, estacion1_y.npy, features.json
                ...
        global/
            ml/
                global_X.npy, global_y.npy, features.json   (ventanas combinadas)
            dl/
                global_X.npy, global_y.npy, features.json
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_ML_BY_STATION = os.path.join(BASE_DIR, "encoded", "ml", "by_station")
INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_BY_STATION = os.path.join(BASE_DIR, "encoded", "dl", "by_station")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_WINDOWS = os.path.join(BASE_DIR, "windows")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_WINDOWS, "by_station", "ml")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_WINDOWS, "by_station", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "dl")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros de ventanas
WINDOW_IN = 72      # horas de entrada
WINDOW_OUT = 72     # horas de salida (predicción)
TARGET_COL = 'O3'   # nombre de la columna objetivo

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def create_windows_from_df(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    """
    Genera ventanas deslizantes a partir de un DataFrame con índice temporal ordenado.
    Retorna:
        X_ml: (n_windows, window_in * n_features)  (aplanado)
        y_ml: (n_windows, window_out)
        X_dl: (n_windows, window_in, n_features)
        y_dl: (n_windows, window_out)
        feature_names: lista de nombres de las columnas (orden original)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index()  # por si acaso
    data = df_sorted.values
    n_features = data.shape[1]
    feature_names = df_sorted.columns.tolist()

    n = len(df_sorted)
    X_ml_list = []
    y_ml_list = []
    X_dl_list = []
    y_dl_list = []

    for i in range(window_in, n - window_out):
        # Ventana de entrada: [i-window_in , i)
        in_data = data[i - window_in:i, :]                # shape (window_in, n_features)
        # Ventana de salida (solo target): [i , i+window_out)
        out_data = df_sorted[target_col].iloc[i:i + window_out].values  # shape (window_out,)
        X_dl_list.append(in_data)
        y_dl_list.append(out_data)
        X_ml_list.append(in_data.flatten())   # aplanado: (window_in * n_features,)
        y_ml_list.append(out_data)

    if len(X_ml_list) == 0:
        print("  Aviso: no se generaron ventanas (serie demasiado corta).")
        return None, None, None, None, feature_names

    X_ml = np.array(X_ml_list, dtype=np.float32)
    y_ml = np.array(y_ml_list, dtype=np.float32)
    X_dl = np.array(X_dl_list, dtype=np.float32)
    y_dl = np.array(y_dl_list, dtype=np.float32)

    return X_ml, y_ml, X_dl, y_dl, feature_names

def process_station_file(file_path, output_ml_dir, output_dl_dir):
    """
    Procesa un archivo CSV de una estación (ya codificado) y guarda las ventanas.
    """
    station_name = Path(file_path).stem
    print(f"  Procesando estación: {station_name}")

    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Crear ventanas
    result = create_windows_from_df(df, target_col=TARGET_COL,
                                    window_in=WINDOW_IN, window_out=WINDOW_OUT)
    if result[0] is None:
        print(f"    No se generaron ventanas para {station_name}")
        return
    X_ml, y_ml, X_dl, y_dl, feature_names = result

    # Guardar archivos ML
    np.save(os.path.join(output_ml_dir, f"{station_name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{station_name}_y.npy"), y_ml)
    with open(os.path.join(output_ml_dir, f"{station_name}_features.json"), 'w') as f:
        json.dump(feature_names, f, indent=2)

    # Guardar archivos DL
    np.save(os.path.join(output_dl_dir, f"{station_name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{station_name}_y.npy"), y_dl)
    with open(os.path.join(output_dl_dir, f"{station_name}_features.json"), 'w') as f:
        json.dump(feature_names, f, indent=2)

    print(f"    Ventanas guardadas: {len(X_ml)} muestras")

def process_global_folder(input_dir_ml, input_dir_dl, output_ml_dir, output_dl_dir):
    """
    Para el caso global: se toman los archivos de todas las estaciones (de las carpetas ml y dl),
    se generan ventanas por separado y se concatenan en un único conjunto.
    input_dir_ml: carpeta con archivos CSV para ML (one-hot)
    input_dir_dl: carpeta con archivos CSV para DL (enteros)
    Ambas deben contener los mismos nombres de estación (uno por estación).
    """
    # Buscar archivos CSV en input_dir_ml (usamos los nombres para iterar)
    ml_files = list(Path(input_dir_ml).glob("*.csv"))
    if not ml_files:
        print(f"  No se encontraron archivos CSV en {input_dir_ml}")
        return

    all_X_ml = []
    all_y_ml = []
    all_X_dl = []
    all_y_dl = []
    feature_names_ml = None
    feature_names_dl = None

    for ml_file in ml_files:
        station_name = ml_file.stem
        dl_file = os.path.join(input_dir_dl, f"{station_name}.csv")
        if not os.path.exists(dl_file):
            print(f"  Aviso: no existe el archivo DL para {station_name}, se omite.")
            continue

        # Cargar ML
        df_ml = pd.read_csv(ml_file, index_col=0, parse_dates=True)
        # Cargar DL
        df_dl = pd.read_csv(dl_file, index_col=0, parse_dates=True)

        # Verificar que los índices sean comparables
        # (deberían ser iguales porque provienen de la misma estación)
        if not df_ml.index.equals(df_dl.index):
            print(f"  Aviso: los índices de {station_name} no coinciden entre ML y DL. Se usará ML para features.")
            # En todo caso, usamos el de ML para consistencia
            df_dl = df_dl.reindex(df_ml.index)

        # Generar ventanas para ML (usando df_ml)
        res_ml = create_windows_from_df(df_ml, target_col=TARGET_COL,
                                        window_in=WINDOW_IN, window_out=WINDOW_OUT)
        if res_ml[0] is None:
            continue
        X_ml, y_ml, _, _, feat_ml = res_ml
        if feature_names_ml is None:
            feature_names_ml = feat_ml
        else:
            if feat_ml != feature_names_ml:
                print(f"  Error: las features de {station_name} no coinciden con las anteriores. Se omitirá.")
                continue
        all_X_ml.append(X_ml)
        all_y_ml.append(y_ml)

        # Generar ventanas para DL (usando df_dl)
        res_dl = create_windows_from_df(df_dl, target_col=TARGET_COL,
                                        window_in=WINDOW_IN, window_out=WINDOW_OUT)
        if res_dl[0] is None:
            continue
        X_dl, y_dl, _, _, feat_dl = res_dl
        if feature_names_dl is None:
            feature_names_dl = feat_dl
        else:
            if feat_dl != feature_names_dl:
                print(f"  Error: las features DL de {station_name} no coinciden.")
                continue
        all_X_dl.append(X_dl)
        all_y_dl.append(y_dl)

    if not all_X_ml:
        print("  No se generaron ventanas para ninguna estación.")
        return

    # Concatenar
    X_ml_global = np.concatenate(all_X_ml, axis=0)
    y_ml_global = np.concatenate(all_y_ml, axis=0)
    X_dl_global = np.concatenate(all_X_dl, axis=0)
    y_dl_global = np.concatenate(all_y_dl, axis=0)

    print(f"  Total ventanas globales ML: {X_ml_global.shape[0]}")
    print(f"  Total ventanas globales DL: {X_dl_global.shape[0]}")

    # Guardar
    np.save(os.path.join(output_ml_dir, "global_X.npy"), X_ml_global)
    np.save(os.path.join(output_ml_dir, "global_y.npy"), y_ml_global)
    with open(os.path.join(output_ml_dir, "global_features.json"), 'w') as f:
        json.dump(feature_names_ml, f, indent=2)

    np.save(os.path.join(output_dl_dir, "global_X.npy"), X_dl_global)
    np.save(os.path.join(output_dl_dir, "global_y.npy"), y_dl_global)
    with open(os.path.join(output_dl_dir, "global_features.json"), 'w') as f:
        json.dump(feature_names_dl, f, indent=2)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Creación de ventanas deslizantes (72h in / 72h out)")
    print("=" * 60)

    # 1. Procesar por estación (individual)
    print("\n--- Procesando datos por estación (individual) ---")
    if os.path.exists(INPUT_ML_BY_STATION) and os.path.exists(INPUT_DL_BY_STATION):
        ml_files = list(Path(INPUT_ML_BY_STATION).glob("*.csv"))
        dl_files = list(Path(INPUT_DL_BY_STATION).glob("*.csv"))
        # Intersectar nombres
        ml_names = {f.stem for f in ml_files}
        dl_names = {f.stem for f in dl_files}
        common = ml_names.intersection(dl_names)
        for station in common:
            ml_path = os.path.join(INPUT_ML_BY_STATION, f"{station}.csv")
            dl_path = os.path.join(INPUT_DL_BY_STATION, f"{station}.csv")
            # Para ML usamos el archivo ML, para DL el archivo DL
            # process_station_file espera un archivo y dos directorios de salida,
            # pero aquí tenemos dos archivos distintos. Vamos a procesar por separado:
            # Primero ML
            df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True)
            res_ml = create_windows_from_df(df_ml)
            if res_ml[0] is not None:
                X_ml, y_ml, _, _, feat_ml = res_ml
                np.save(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_X.npy"), X_ml)
                np.save(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_y.npy"), y_ml)
                with open(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_features.json"), 'w') as f:
                    json.dump(feat_ml, f)
                print(f"  ML {station}: {len(X_ml)} ventanas")
            # Luego DL
            df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True)
            res_dl = create_windows_from_df(df_dl)
            if res_dl[0] is not None:
                X_dl, y_dl, _, _, feat_dl = res_dl
                np.save(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_X.npy"), X_dl)
                np.save(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_y.npy"), y_dl)
                with open(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_features.json"), 'w') as f:
                    json.dump(feat_dl, f)
                print(f"  DL {station}: {len(X_dl)} ventanas")
    else:
        print("  Las carpetas de entrada por estación no existen. Se omite.")

    # 2. Procesar global (todas las estaciones juntas)
    print("\n--- Procesando datos globales ---")
    if os.path.exists(INPUT_ML_GLOBAL) and os.path.exists(INPUT_DL_GLOBAL):
        process_global_folder(INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                              OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)
    else:
        print("  Las carpetas de entrada global no existen. Se omite.")

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_WINDOWS}")
